# SHAP Value Feature Selection

## Imports

In [27]:
import os
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
try:
    import shap
except ModuleNotFoundError:
    ! pip install shap
    import shap

## Constants

In [9]:
str_project = os.getcwd().split('\\')[4]
print(f'Project: {str_project}')
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
str_dirname_output = './output'
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

Project: 20240509_christian_internship
Task: 12_continued
Subtask: 01_feature_selection


## Read

In [36]:
str_filename = "train.csv"
str_local_path = f'./input/{str_filename}'
df = pd.read_csv(str_local_path)

In [37]:
# missing values
missing_values = df.isnull().sum() / len(df)

# threshold for missing values
threshold = 0.65

# drop features with missing rate higher than threshold
df = df.loc[:, missing_values <= threshold]

In [38]:
# separate features and target
X = df.drop('TARGET', axis=1)
y = df['TARGET']

# define imputer for numerical and categorical features
num_imputer = SimpleImputer(strategy='mean')
cat_imputer = SimpleImputer(strategy='most_frequent')

# separate numerical and categorical features
num_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(include=['object']).columns

# impute numerical features
X[num_features] = num_imputer.fit_transform(X[num_features])

# impute categorical features
X[cat_features] = cat_imputer.fit_transform(X[cat_features])

In [39]:
# initialize
model = CatBoostClassifier(iterations=100, depth=5, learning_rate=0.1, verbose=False)

# train the initial model
cat_features = cat_features.tolist()
model.fit(X, y, cat_features=cat_features)

# pred probs for the initial model
probabilities_initial = model.predict_proba(X)[:, 1]

In [40]:
# get SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# calculate mean absolute SHAP values for feature importance
shap_importance = pd.DataFrame(list(zip(X.columns, np.abs(shap_values).mean(axis=0))), columns=['Feature', 'SHAP Importance'])

# sort features by SHAP importance
shap_importance = shap_importance.sort_values(by='SHAP Importance', ascending=False)

# select top features (top 30 features in this case)
top_features = shap_importance['Feature'].head(30).tolist()

# reduce dataset to top features
X_selected = X[top_features]

In [47]:
# initialize and train the final CatBoost model
final_model = CatBoostClassifier(iterations=100, depth=5, learning_rate=0.3337, verbose=False)
final_model.fit(X_selected, y, cat_features=[top_features.index(f) for f in cat_features if f in top_features])

# pred probs for the final model
probabilities_final = final_model.predict_proba(X_selected)[:, 1]

In [46]:
# calc ROC AUC for the initial model
roc_auc_initial = roc_auc_score(y, probabilities_initial)
print(f'ROC AUC before feature selection: {roc_auc_initial}')

# calc ROC AUC for the final model
roc_auc_final = roc_auc_score(y, probabilities_final)
print(f'ROC AUC after feature selection: {roc_auc_final}')

ROC AUC before feature selection: 0.7619334875545754
ROC AUC after feature selection: 0.763388244126456


# LR Coef Feature Selection

In [48]:
# read
str_file_path = f'./input/{str_filename}'
df = pd.read_csv(str_file_path)

In [49]:
# missing rate 65% threshold
missing_rate = df.isnull().mean()
features_to_keep = missing_rate[missing_rate <= 0.65].index
df_filtered = df[features_to_keep]

# imputation
numerical_cols = df_filtered.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df_filtered.select_dtypes(include=['object']).columns

imputer_num = SimpleImputer(strategy='mean')
df_filtered[numerical_cols] = imputer_num.fit_transform(df_filtered[numerical_cols])

imputer_cat = SimpleImputer(strategy='most_frequent')
df_filtered[categorical_cols] = imputer_cat.fit_transform(df_filtered[categorical_cols])

C:\Users\ctsvetanov\AppData\Local\Temp\ipykernel_21124\2175638066.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered[numerical_cols] = imputer_num.fit_transform(df_filtered[numerical_cols])
C:\Users\ctsvetanov\AppData\Local\Temp\ipykernel_21124\2175638066.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered[categorical_cols] = imputer_cat.fit_transform(df_filtered[categorical_cols])


In [50]:
# LR to select important features
df_encoded = pd.get_dummies(df_filtered, columns=categorical_cols, drop_first=True)
X = df_encoded.drop('TARGET', axis=1)
y = df_encoded['TARGET']
# scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# rlr
lasso = Lasso(alpha=0.01)
lasso.fit(X_scaled, y)
important_features = X.columns[lasso.coef_ != 0]

In [69]:
# build catboost model
X_selected = X[important_features]
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)
model = CatBoostClassifier(learning_rate=0.025, verbose=0, random_state=42)
model.fit(X_train, y_train)
y_pred_proba = model.predict_proba(X_test)[:, 1]

In [70]:
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {roc_auc}")
# logloss
logloss = log_loss(y_test, y_pred_proba)
print(f'Log Loss: {logloss}')

ROC AUC Score: 0.7393940409654209
Log Loss: 0.2515414456590706
